In [ ]:
prefix_path = '../..'

import sys
sys.path.append(prefix_path)

import logging
import os
import random
from collections import Counter
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from egh_vlm.hallu_dataset import HalluDataset, load_hallu_dataset, hallu_collate_fn, split_stratified
from egh_vlm.hallu_ffn_detector import HalluFFNDetector, train_ffn_detector, eval_ffn_detector
from egh_vlm.utils import Qwen3ModelBundle, load_phd_dataset

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_FOLDER_PATH = f'{prefix_path}/data/phd/features/linear_probing'
OUTPUT_PATH = f'{prefix_path}/data/phd/evaluations/evaluation_egh_vlm_phd_sample_full_context_qwen3_linear_probing.json'
LOGGING_PATH = f'{prefix_path}/data/logs/log_egh_vlm_phd_sample_full_context_qwen3.txt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
TRAIN_RATIO = 0.7

#### Load Dataset

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH,
)
print(f'Length of dataset: {len(dataset)}')

features = {}

for f in sorted(os.listdir(FEATURES_FOLDER_PATH)):
    if not f.endswith('.pt'):
        continue
    layer = int(f[len('features_phd_sample_full_context_qwen3_layer'):-3])
    path = os.path.join(FEATURES_FOLDER_PATH, f) 
    features[layer] = load_hallu_dataset(path) if os.path.isfile(path) else None
    
print(f'Loaded feature layers: {sorted(features.keys())}')
hidden_size = features[8].embs[0].size(-1) if len(features) > 0 else 0
print(f'Hidden size of features: {hidden_size}')

#### Analysis

In [ ]:
def train(
    train_dataloader: DataLoader,
    test_dataloader: DataLoader,
    hidden_size: int,
    logging_path: str,
    device: torch.device,
    train_ratio: float=0.7,
    epochs_count: int=30,
    lr: float=1e-4,
    features_weight: float=0.5,
):  
    logging.basicConfig(filename=logging_path, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    logging.info(f'Starting training with hidden_size={hidden_size}, train_ratio={train_ratio}, epochs_count={epochs_count}, lr={lr}, features_weight={features_weight}')
    
    result = {
        'train_ratio': train_ratio,
        'epochs_count': epochs_count,
        'lr': lr,
        'features_weight': features_weight,
        'acc': {
            'value': 0.0,
            'epoch': -1
        },
        'f1': {
            'value': 0.0,
            'epoch': -1
        },
        'pr_auc': {
            'value': 0.0,
            'epoch': -1
        }
    }
    detector = HalluFFNDetector(
        input_dim=hidden_size, 
        hidden_dim=hidden_size, 
        output_dim=1,
        device=device,
        w=features_weight)
    loss_fn = nn.BCELoss()
    optim = torch.optim.Adam(detector.parameters(), lr=lr)
    
    for i in range(epochs_count):
        loss = train_ffn_detector(detector, loss_fn, optim, train_dataloader)
        eval_metrics = eval_ffn_detector(detector, test_dataloader)
        acc, f1, pr_auc = eval_metrics['acc'], eval_metrics['f1'], eval_metrics['pr_auc']
        
        logging.info(f'Epoch {i+1}/{epochs_count}, Loss: {loss:.4f}')
        logging.info(f'Epoch {i+1}/{epochs_count}, Accuracy: {acc:.4f}, F1: {f1:.4f}, PR-AUC: {pr_auc:.4f}\n')
        
        if acc > result['acc']['value']:
            result['acc']['value'] = acc
            result['acc']['epoch'] = i + 1
        if f1 > result['f1']['value']:
            result['f1']['value'] = f1
            result['f1']['epoch'] = i + 1
        if pr_auc > result['pr_auc']['value']:
            result['pr_auc']['value'] = pr_auc
            result['pr_auc']['epoch'] = i + 1

    logging.info(f'Best Accuracy: {result["acc"]["value"]:.4f} at epoch {result["acc"]["epoch"]}')
    logging.info(f'Best F1: {result["f1"]["value"]:.4f} at epoch {result["f1"]["epoch"]}')
    logging.info(f'Best PR-AUC: {result["pr_auc"]["value"]:.4f} at epoch {result["pr_auc"]["epoch"]}')
    
    # Clean up
    logger = logging.getLogger()
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)
    return detector, result

In [ ]:
results = []

for layer in features:
    feature = features[layer]
    train_dataset, test_dataset = split_stratified(feature, train_ratio=TRAIN_RATIO, random_state=42)
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=32,
        collate_fn=hallu_collate_fn,
        shuffle=True,
    )
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=32,
        collate_fn=hallu_collate_fn,
        shuffle=True,
    )
    detector, result = train(
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        hidden_size=hidden_size,
        logging_path=LOGGING_PATH.replace('.txt', f'_layer{layer}.txt'),
        train_ratio=TRAIN_RATIO,
        device=DEVICE,
    )
    results.append(result)
    
with open(OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)